# Motion and TrackingCovers sparse and dense optical flow, background subtraction, and CAMShift object tracking in video.

## Import libraries

In [1]:
# import Open CV library
import cv2

import matplotlib.pyplot as plt
import numpy as np

## Optical Flow
Optical flow describes the apparent motion of pixels between consecutive frames in a video. Optical flow methods are generally divided into two main categories: `sparse` optical flow and `dense` optical flow. 
- `Sparse` Optical Flow: Tracks only selected points(corners, features, Shi-Tomasi points).<br> 
    &nbsp;Example: `Lucas-Kanade`
    
- `Dense` Optical Flow: Computes motion for every pixel.<br>
    &nbsp;Example: `Farneback`, `DeepFlow`, `RAFT` (deep learning)

### Lucas-Kanade


In [26]:

cap = cv2.VideoCapture("videos/snufkin.mp4")

# Shi-Tomasi parameters
feature_params = dict(
    maxCorners=100,
    qualityLevel=0.3,
    minDistance=7,
    blockSize=7
)

# Lucas-Kanade parameters
lk_params = dict(
    winSize=(15,15),
    maxLevel=2,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        10,
        0.03
    )
)

# Initial frame
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Detect initial corners
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# Mask for drawing tracks
mask = np.zeros_like(old_frame)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Compute optical flow
    p1, st, err = cv2.calcOpticalFlowPyrLK(
        old_gray,
        frame_gray,
        p0,
        None,
        **lk_params
    )

    # Select good points
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw motion tracks
    for new, old in zip(good_new, good_old):

        a, b = new.ravel() # (x1,y1)
        c, d = old.ravel() # (x2,y2)

        mask = cv2.line(
            mask,
            (int(a), int(b)),
            (int(c), int(d)),
            (0,255,0),
            2
        )

        frame = cv2.circle(frame, (int(a), int(b)), 5, (0,0,255), -1)

    output = cv2.add(frame, mask)

    cv2.imshow("Optical Flow", output)

    if cv2.waitKey(30) & 0xFF == 27:
        break

    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1,1,2)

cap.release()
cv2.destroyAllWindows()


### Farneback


In [22]:

cap = cv2.VideoCapture("videos/snufkin.mp4")

# Initial frame
ret, frame1 = cap.read()

old_frame = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

# HSV image for visualization
hsv = np.zeros_like(frame1)
hsv[...,1] = 255

while True:

    ret, frame2 = cap.read()

    if not ret:
        break

    next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Compute dense optical flow
    flow = cv2.calcOpticalFlowFarneback(
        old_frame,
        next_frame,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0
    )

    # Convert flow vectors to magnitude and angle
    magnitude, angle = cv2.cartToPolar(
        flow[...,0],
        flow[...,1]
    )

    # Hue represents direction
    hsv[...,0] = angle * 180 / (2*np.pi)

    # Value represents magnitude
    hsv[...,2] = cv2.normalize(
        magnitude,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    # Convert HSV to BGR for display
    rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    cv2.imshow("Dense Optical Flow", rgb)

    if cv2.waitKey(30) & 0xFF == 27:
        break

    old_frame = next_frame

cap.release()
cv2.destroyAllWindows()


## Background Subtraction
It works by modeling the background of a scene and comparing each incoming frame to that model. Any significant difference between the current frame and the background is classified as foreground.

In [21]:

cap = cv2.VideoCapture("videos/little-my.mp4")

# Create background subtractor
fgbg = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=16,
    detectShadows=True
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # Apply background subtraction
    fgmask = fgbg.apply(frame)

    cv2.imshow("Original", frame)
    cv2.imshow("Foreground Mask", fgmask)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


## CAMShift
CAMShift(Continuously Adaptive Mean Shift) is a classical **object tracking** algorithm used in computer vision for tracking moving objects in video streams. It is an extension of the Mean Shift algorithm and was designed to handle changes in object size, orientation, and position  over time.

**Core idea**:
 
Select object → Compute color histogram → Back projection → Apply Mean Shift → Update window size → Repeat for next frame



In [18]:
cap = cv2.VideoCapture("videos/snufkin.mp4")

ret, frame = cap.read() # First frame

# 1- Select object
x, y, w, h = 148, 123, 28, 40
track_window = (x, y, w, h)
roi = frame[y:y+h, x:x+w]

# 2- Compute histogram
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv_roi, np.array((0., 60., 32.)), np.array((180., 255., 255.)))
roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0,180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX) # Normalize

# Termination criteria
term_crit = (
    cv2.TERM_CRITERIA_EPS |
    cv2.TERM_CRITERIA_COUNT,
    10,
    1
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 3- Back projection
    dst = cv2.calcBackProject(
        [hsv],
        [0],
        roi_hist,
        [0,180],
        1
    )

    # 4- Apply CAMShift & Update window size
    ret, track_window = cv2.CamShift(
        dst,
        track_window,
        term_crit
    )

    # Draw rotated rectangle
    pts = cv2.boxPoints(ret)
    pts = pts.astype(int)

    result = cv2.polylines(
        frame,
        [pts],
        True,
        (0,255,0),
        2
    )

    cv2.imshow("CAMShift Tracking", result)

    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
